In [ ]:
# @title 🚀 1-Click FreeFakeStudio Engine
# @markdown ### 👇 Tick the models you want to download:
Z_Image_Turbo = True  # @param {type:"boolean"}
FLUX_Klein_4B = False  # @param {type:"boolean"}
FLUX_Klein_9B = False  # @param {type:"boolean"}
Qwen_Image_Edit = False  # @param {type:"boolean"}
ERNIE_Image_Turbo = False  # @param {type:"boolean"}

import os, sys, subprocess, time, re, glob
import torch
from IPython.display import display, HTML, clear_output

# ── 1. Check GPU ──────────────────────────────────────────
if not torch.cuda.is_available():
    clear_output()
    print("❌ GPU required. Please click Runtime -> Change runtime type -> T4 GPU -> Save, then click Play again.")
    raise SystemExit(1)

if not any([Z_Image_Turbo, FLUX_Klein_4B, FLUX_Klein_9B, Qwen_Image_Edit, ERNIE_Image_Turbo]):
    clear_output()
    print("❌ Please tick at least one model checkbox above!")
    raise SystemExit(1)

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU Connected: {gpu_name}")
print("⏳ Step 1/3: Setting up ComfyUI backend & dependencies...")

# ── 2. ComfyUI & Deps ─────────────────────────────────────
need_gguf = any([Qwen_Image_Edit, FLUX_Klein_9B, ERNIE_Image_Turbo])
need_diffusers = Qwen_Image_Edit

if not os.path.exists('/content/ComfyUI'):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", "/content/ComfyUI"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/requirements.txt"], check=True)

if need_gguf and not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    subprocess.run(["git", "clone", "https://github.com/city96/ComfyUI-GGUF.git", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt"], check=True)

if need_diffusers:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/diffusers"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg[gpu]", "onnxruntime-gpu", "aria2", "gradio"], check=True)

# ── 3. Model Dirs & Downloads ─────────────────────────────
DIFF   = '/content/ComfyUI/models/diffusion_models'
CLIP   = '/content/ComfyUI/models/clip'
TXTENC = '/content/ComfyUI/models/text_encoders'
VAE    = '/content/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if os.path.exists(path) and os.path.getsize(path) > 1024:
        print(f'  ✅ {name} (cached)')
    else:
        if os.path.exists(path):
            os.remove(path)
        print(f'  ⏳ Downloading {name}...')
        subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", dest_dir, "-o", name], check=True)
        print(f'  ✅ {name} ready!')

print("\n📦 Step 2/3: Downloading selected models...")

if Z_Image_Turbo:
    print("\n⚡ Z-Image Turbo:")
    dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

if FLUX_Klein_4B or FLUX_Klein_9B:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

if FLUX_Klein_4B:
    print("\n🌊 FLUX.2-klein 4B:")
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b.safetensors', DIFF, 'flux-2-klein-4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors', TXTENC, 'qwen_3_4b_fp4_flux2.safetensors')

if FLUX_Klein_9B:
    print("\n🔮 FLUX.2-klein 9B:")
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors', DIFF, 'flux-2-klein-9b-kv-fp8.safetensors')
    dl('https://huggingface.co/unsloth/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q2_K_L.gguf', TXTENC, 'Qwen3-8B-Q2_K_L.gguf')

if Qwen_Image_Edit:
    print("\n🎨 Qwen-Image-Edit:")
    dl('https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q3_K_M.gguf', DIFF, 'qwen-image-edit-2511-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/mmproj-F16.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf')
    dl('https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

if ERNIE_Image_Turbo:
    print("\n🖌️ ERNIE-Image Turbo:")
    dl('https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/resolve/main/ernie-image-turbo-Q6_K.gguf', DIFF, 'ernie-image-turbo-Q6_K.gguf')
    dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/text_encoders/ministral-3-3b.safetensors', TXTENC, 'ministral-3-3b.safetensors')
    if not (FLUX_Klein_4B or FLUX_Klein_9B):
        dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

# Symlink clip/ <-> text_encoders/
for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)
for f in glob.glob(f'{TXTENC}/*'):
    link = os.path.join(CLIP, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)

# ── 4. Pull FreeFakeStudio & Start ────────────────────────
print("\n🚀 Step 3/3: Launching FreeFakeStudio Engine...")
REPO = 'https://github.com/MuntasirMalek/FreeFakeStudio.git'
if os.path.exists('/content/FreeFakeStudio'):
    os.chdir('/content/FreeFakeStudio')
    subprocess.run(["git", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO, "/content/FreeFakeStudio"], check=True)
    os.chdir('/content/FreeFakeStudio')

proc = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

gradio_url = None
url_pattern = re.compile(r'(https://[a-zA-Z0-9-]+\.gradio\.live)')

for line in proc.stdout:
    match = url_pattern.search(line)
    if match:
        gradio_url = match.group(1)
        break

if gradio_url:
    web_app_url = f"https://muntasirmalek.github.io/FreeFakeStudio/?backend={gradio_url}"
    clear_output()
    display(HTML(f'''
    <div style="text-align: center; padding: 40px 10px;">
        <a href="{web_app_url}" target="_blank" style="display: inline-block; background: #2563eb; color: #ffffff; padding: 22px 60px; border-radius: 18px; font-size: 28px; font-weight: 900; text-decoration: none; box-shadow: 0 12px 35px rgba(37,99,235,0.45);">
            🚀 OPEN STUDIO 🚀
        </a>
    </div>
    '''))

try:
    for line in proc.stdout:
        pass
except KeyboardInterrupt:
    proc.terminate()
